# Chuẩn bị data v2

Giữ nguyên ID/split nguồn. Data v2 phải có fingerprint preprocessing.
Train và chọn CNN chỉ dùng train/validation. Notebook finalize mới đánh giá test.
Output cũ là baseline lịch sử, không ghi đè. `SMOKE_TEST=True` không phải kết quả đầy đủ.
Gắn dataset/Notebook Output phù hợp và runtime assets (detector + pretrained weights).


In [ ]:
from pathlib import Path
import json, os
SEARCH_ROOT = Path('/kaggle/input')
def input_files(name):
    matches=[]
    for directory,dirs,files in os.walk(SEARCH_ROOT):
        dirs[:]=[d for d in dirs if d not in {'images','__pycache__','.git'}]
        if name in files:matches.append(Path(directory)/name)
    return matches
def find_dataset(derived):
    candidates=[]
    for p in input_files('preparation_report.json'):
        if p.parent.name=='metadata' and ('preprocessing' in json.loads(p.read_text()))==derived:
            candidates.append(p.parent.parent)
    if len(candidates)!=1:raise ValueError(f'Set DATA_ROOT explicitly; candidates: {candidates}')
    return candidates[0]
def find_asset(name):
    matches=input_files(name)
    if len(matches)!=1:raise ValueError(f'Expected one {name}; got {matches}')
    return matches[0]
DATA_ROOT = None
OUTPUT_ROOT = Path('/kaggle/working/hand_gesture_v2')
DEVICE = 'auto'
SMOKE_TEST = False
NUM_WORKERS = 0
WEIGHTS_PATH = None  # MobileNet IMAGENET1K_V1 .pth; no download/fallback during training
import os
if Path('/kaggle').exists():
    os.environ['HG_KAGGLE']='1'
    os.environ['HG_REQUIRE_CUDA']='1'
    os.environ.setdefault('HG_PROGRESS_PATH','/kaggle/working/progress.json')

DETECTOR_PATH = None


In [ ]:
import sys, types, os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
for package in ['handgesture','scripts']:
    module=types.ModuleType(package);module.__path__=[];sys.modules[package]=module


In [ ]:
# Shared local source: handgesture/runtime.py
_module = types.ModuleType('handgesture.runtime')
_module.__file__ = 'handgesture/runtime.py'
sys.modules['handgesture.runtime'] = _module
_source = '"""Small shared runtime controls for observable, bounded Kaggle execution."""\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport codecs\nimport json\nimport os\nimport selectors\nimport signal\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=path.name+\'.\', suffix=\'.tmp\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'w\') as stream:\n            json.dump(value, stream, ensure_ascii=False, allow_nan=False, indent=2)\n            stream.write(\'\\n\')\n        os.replace(temporary, path)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef progress(**fields):\n    """Only actual milestones call this; a watchdog heartbeat is not progress."""\n    path = os.environ.get(\'HG_PROGRESS_PATH\')\n    value = {}\n    if path and Path(path).exists():\n        value = json.loads(Path(path).read_text())\n    value.update(fields)\n    value.update(last_progress_at_utc=utc_now(), pid=os.getpid())\n    if path:\n        atomic_json(path, value)\n    line=json.dumps(value, ensure_ascii=False, allow_nan=False)\n    event_log=os.environ.get(\'HG_EVENT_LOG_PATH\')\n    if event_log and os.environ.get(\'HG_EVENT_LOG_OWNER_PID\')==str(os.getpid()):\n        event_path=Path(event_log);event_path.parent.mkdir(parents=True,exist_ok=True)\n        with event_path.open(\'a\') as stream:stream.write(line+\'\\n\')\n    print(line, flush=True)\n    return value\n\n\n@contextmanager\ndef stage(name, model=None):\n    start = time.monotonic()\n    progress(stage=name, model=model, status=\'running\', completed=0, total=None,\n             epoch=None, error=None, started_at_utc=utc_now())\n    try:\n        yield\n    except BaseException as exc:\n        progress(stage=name, model=model, status=\'timeout\' if isinstance(exc,TimeoutError) else \'error\',\n                 error=f\'{type(exc).__name__}: {exc}\', elapsed_seconds=time.monotonic()-start)\n        raise\n    else:\n        progress(stage=name, model=model, status=\'complete\', elapsed_seconds=time.monotonic()-start)\n\n\ndef validate_workers(workers):\n    if workers < 0:\n        raise ValueError(\'workers must be nonnegative\')\n    if os.environ.get(\'HG_KAGGLE\') == \'1\' and workers:\n        raise ValueError(\'Kaggle runtime requires workers=0; parallel workers are not supported safely\')\n\n\ndef require_cuda():\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is required for Kaggle neural-network training; refusing CPU fallback\')\n    # Test allocation, kernel execution and synchronization, not only driver discovery.\n    x = torch.ones((8,8), device=\'cuda\')\n    result = x @ x\n    torch.cuda.synchronize()\n    if not bool(torch.all(result == 8).item()):\n        raise RuntimeError(\'CUDA compute preflight failed\')\n    progress(gpu=torch.cuda.get_device_name(), cuda=torch.version.cuda)\n\n\ndef _stop_group(process):\n    """Kill descendants too, even if the group leader has already exited."""\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        pass\n    time.sleep(.2)\n    try:\n        os.killpg(process.pid, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait()\n\n\ndef run_logged(command, log_path, deadline, env=None):\n    """Tee combined output immediately; deadline is absolute monotonic time."""\n    if time.monotonic() >= deadline:\n        raise TimeoutError(\'Experiment deadline exceeded before command start\')\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               start_new_session=True, env=env)\n    decoder = codecs.getincrementaldecoder(\'utf-8\')(errors=\'replace\')\n    try:\n        with log_path.open(\'a\', buffering=1) as log, selectors.DefaultSelector() as selector:\n            selector.register(process.stdout, selectors.EVENT_READ)\n            while selector.get_map() or process.poll() is None:\n                remaining = deadline-time.monotonic()\n                if remaining <= 0:\n                    raise TimeoutError(\'Experiment deadline exceeded; terminating subprocess group\')\n                for key, _ in selector.select(min(.1,remaining)):\n                    chunk = os.read(key.fileobj.fileno(),65536)\n                    if not chunk:\n                        selector.unregister(key.fileobj)\n                        continue\n                    value = decoder.decode(chunk)\n                    sys.stdout.write(value);sys.stdout.flush()\n                    log.write(value);log.flush()\n            tail = decoder.decode(b\'\',final=True)\n            if tail:\n                sys.stdout.write(tail);sys.stdout.flush();log.write(tail)\n        code = process.wait()\n        if code:\n            raise subprocess.CalledProcessError(code, command)\n    except BaseException:\n        _stop_group(process)\n        raise\n    finally:\n        process.stdout.close()\n'
exec(compile(_source, 'handgesture/runtime.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/preprocessing.py
_module = types.ModuleType('handgesture.preprocessing')
_module.__file__ = 'handgesture/preprocessing.py'
sys.modules['handgesture.preprocessing'] = _module
_source = '"""The same geometry is used for Kaggle images and unmirrored webcam frames."""\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\n@dataclass(frozen=True)\nclass PreprocessingConfig:\n    detector_sha256: str\n    version: str = \'pixel_wrist_max3d_square_v2\'\n    detection_confidence: float = .5\n    presence_confidence: float = .5\n    tracking_confidence: float = .5\n    roi_scale: float = 1.4\n    epsilon: float = 1e-6\n    num_hands: int = 1\n    mirror_input: bool = False\n    detector_input_padding: float = .5\n    detector_runtime_version: str = \'0.10.21\'\n\n    def __post_init__(self):\n        if (self.version != \'pixel_wrist_max3d_square_v2\' or self.roi_scale < 1\n                or self.epsilon <= 0 or self.num_hands != 1 or self.mirror_input\n                or not 0 <= self.detector_input_padding <= 1\n                or any(not 0 <= v <= 1 for v in (self.detection_confidence,\n                       self.presence_confidence, self.tracking_confidence))):\n            raise ValueError(\'Invalid preprocessing configuration\')\n\n    @property\n    def fingerprint(self):\n        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True,\n                                        allow_nan=False).encode()).hexdigest()\n\n    def assert_matches(self, fingerprint):\n        if fingerprint != self.fingerprint:\n            raise ValueError(\'Preprocessing fingerprint mismatch; retrain with this pipeline\')\n\n\ndef normalize_landmarks(landmarks, width, height, epsilon=1e-6):\n    points = np.asarray(landmarks, dtype=np.float64)\n    if points.shape != (21, 3) or not np.isfinite(points).all() or min(width, height) <= 0:\n        raise ValueError(\'Invalid landmarks or image dimensions\')\n    pixel = points * np.array([width, height, width])\n    relative = pixel - pixel[0]\n    scale = np.linalg.norm(relative, axis=1).max()\n    if not np.isfinite(scale) or scale <= epsilon:\n        raise ValueError(\'Degenerate landmark scale\')\n    return (relative / scale).astype(np.float32).reshape(63)\n\n\ndef square_roi(image, landmarks, roi_scale=1.4):\n    height, width = image.shape[:2]\n    points = np.asarray(landmarks, dtype=float)\n    if points.shape != (21, 3) or not np.isfinite(points).all():\n        raise ValueError(\'Invalid ROI landmarks\')\n    xy = points[:, :2] * [width, height]\n    low, high = xy.min(axis=0), xy.max(axis=0)\n    side = int(math.ceil(float((high-low).max()) * roi_scale))\n    if side < 1 or side > 4 * max(width, height):\n        raise ValueError(\'Invalid ROI extent\')\n    center = (low+high)/2\n    left, top = np.floor(center-side/2).astype(int)\n    right, bottom = left+side, top+side\n    sx0, sy0, sx1, sy1 = max(0,left), max(0,top), min(width,right), min(height,bottom)\n    if sx1 <= sx0 or sy1 <= sy0:\n        raise ValueError(\'ROI outside image\')\n    roi = np.zeros((side,side,3), dtype=np.uint8)\n    roi[sy0-top:sy1-top, sx0-left:sx1-left] = image[sy0:sy1,sx0:sx1]\n    return roi, (int(left),int(top),int(right),int(bottom))\n\n\n@dataclass\nclass HandResult:\n    status: str\n    features: object = None\n    roi: object = None\n    landmarks: object = None\n    bbox: object = None\n    reason: str = \'\'\n\n\nclass HandPreprocessor:\n    def __init__(self, config, model_path=None, mode=\'IMAGE\', detector=None):\n        self.config = config\n        self._landmarker = None\n        self._last_timestamp = -1\n        self.mode = mode\n        if detector is not None:\n            self._detect = detector\n            return\n        if mode not in {\'IMAGE\',\'VIDEO\'}:\n            raise ValueError(\'mode must be IMAGE or VIDEO\')\n        if sha256(model_path) != config.detector_sha256:\n            raise ValueError(\'Detector SHA-256 mismatch\')\n        from importlib.metadata import version\n        if version(\'mediapipe\') != config.detector_runtime_version:\n            raise ValueError(\'MediaPipe runtime differs from fingerprint; use the documented v2 environment\')\n        import mediapipe as mp\n        from mediapipe.tasks.python import BaseOptions, vision\n        self._mp = mp\n        options = vision.HandLandmarkerOptions(\n            base_options=BaseOptions(model_asset_path=str(model_path),delegate=BaseOptions.Delegate.CPU),\n            running_mode=getattr(vision.RunningMode,mode), num_hands=config.num_hands,\n            min_hand_detection_confidence=config.detection_confidence,\n            min_hand_presence_confidence=config.presence_confidence,\n            min_tracking_confidence=config.tracking_confidence)\n        self._landmarker = vision.HandLandmarker.create_from_options(options)\n        self._detect = self._mediapipe_detect\n\n    def _mediapipe_detect(self, image, timestamp_ms):\n        value = self._mp.Image(image_format=self._mp.ImageFormat.SRGB,\n                               data=np.ascontiguousarray(image))\n        if self.mode == \'VIDEO\':\n            timestamp_ms = max(int(timestamp_ms), self._last_timestamp+1)\n            self._last_timestamp = timestamp_ms\n            result = self._landmarker.detect_for_video(value,timestamp_ms)\n        else:\n            result = self._landmarker.detect(value)\n        if not result.hand_landmarks:\n            return None\n        return np.array([[p.x,p.y,p.z] for p in result.hand_landmarks[0]])\n\n    def detect(self, image, timestamp_ms=0):\n        if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:\n            raise ValueError(\'Expected uint8 RGB image HxWx3\')\n        height,width=image.shape[:2]\n        pad_x,pad_y=int(width*self.config.detector_input_padding),int(height*self.config.detector_input_padding)\n        padded=np.pad(image,((pad_y,pad_y),(pad_x,pad_x),(0,0)))\n        points=self._detect(padded,timestamp_ms)\n        return None if points is None else self.map_detector_landmarks(points,width,height)\n\n    def map_detector_landmarks(self,points,width,height):\n        points=np.asarray(points,dtype=float).copy()\n        if points.shape!=(21,3):raise ValueError(\'Invalid detector landmark shape\')\n        pad_x,pad_y=int(width*self.config.detector_input_padding),int(height*self.config.detector_input_padding)\n        points[:,0]=(points[:,0]*(width+2*pad_x)-pad_x)/width\n        points[:,1]=(points[:,1]*(height+2*pad_y)-pad_y)/height\n        points[:,2]*=(width+2*pad_x)/width\n        return points\n\n    def transform(self, image, points):\n        if points is None:\n            return HandResult(\'no_hand\', reason=\'no_hand\')\n        try:\n            features = normalize_landmarks(points,image.shape[1],image.shape[0],self.config.epsilon)\n            roi,bbox = square_roi(image,points,self.config.roi_scale)\n        except ValueError as exc:\n            return HandResult(\'invalid\',reason=str(exc))\n        return HandResult(\'ok\',features,roi,np.asarray(points),bbox)\n\n    def process(self, image, timestamp_ms=0):\n        return self.transform(image,self.detect(image,timestamp_ms))\n\n    def close(self):\n        if self._landmarker is not None:\n            self._landmarker.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self,*args):\n        self.close()\n'
exec(compile(_source, 'handgesture/preprocessing.py', 'exec'), _module.__dict__)

In [ ]:
# Shared local source: handgesture/data.py
_module = types.ModuleType('handgesture.data')
_module.__file__ = 'handgesture/data.py'
sys.modules['handgesture.data'] = _module
_source = '"""Prepare a separate paired dataset; never reassign the source split."""\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\n\nfrom handgesture.preprocessing import HandPreprocessor, PreprocessingConfig, sha256\nfrom handgesture.runtime import progress, validate_workers\n\nGESTURES = [\'fist\',\'one\',\'peace\',\'three\',\'four\',\'palm\']\nMAPPING = [{\'class_id\': i, \'gesture\': g} for i,g in enumerate(GESTURES)]\nLM_COLUMNS = [f\'lm_{i}\' for i in range(63)]\nSPLITS = [\'train\',\'validation\',\'test\']\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\')\n\n\ndef safe_path(root, relative):\n    root = Path(root).resolve()\n    path = (root/relative).resolve()\n    if not path.is_relative_to(root) or not path.is_file():\n        raise ValueError(f\'Invalid or missing path: {relative}\')\n    return path\n\n\ndef _extract_record(row,source_root,stage,processor):\n    item = {\'sample_id\':row[\'sample_id\'],\'split\':row[\'split\'],\n            \'class_id\':int(row[\'class_id\']),\'gesture\':row[\'gesture\']}\n    path = safe_path(source_root,row[\'image_path\'])\n    if row.get(\'image_sha256\') and sha256(path) != row[\'image_sha256\']:\n        raise ValueError(f\'Source image hash mismatch: {path}\')\n    try:\n        with Image.open(path) as image:\n            rgb = np.array(ImageOps.exif_transpose(image).convert(\'RGB\'))\n        result = processor.process(rgb)\n        item[\'status\'] = result.status\n        item[\'reason\'] = result.reason\n        if result.status == \'ok\':\n            # ID-derived names avoid collisions and path traversal.\n            name = __import__(\'hashlib\').sha256(row[\'sample_id\'].encode()).hexdigest()+\'.png\'\n            relative = f"images/{row[\'split\']}/{row[\'gesture\']}/{name}"\n            destination = stage/relative\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            Image.fromarray(result.roi).save(destination)\n            item.update(image_path=relative,image_sha256=sha256(destination),\n                        width=result.roi.shape[1],height=result.roi.shape[0],\n                        source_image_path=row[\'image_path\'],source_image_sha256=sha256(path),\n                        source_label=row.get(\'source_label\',\'train_val_\'+row[\'gesture\']),\n                        source_filepath=row.get(\'source_filepath\',row[\'image_path\']),\n                        roi_bbox=json.dumps(result.bbox),\n                        **dict(zip(LM_COLUMNS,map(float,result.features))))\n    except (OSError,ValueError) as exc:\n        item.update(status=\'invalid\',reason=f\'{type(exc).__name__}: {exc}\')\n    return item\n\n\n_WORKER_CONTEXT = None\n\n\ndef _initialize_worker(config,model_path,source_root,stage,processor):\n    global _WORKER_CONTEXT\n    detector=processor or HandPreprocessor(config,model_path,mode=\'IMAGE\')\n    _WORKER_CONTEXT=(source_root,stage,detector)\n\n\ndef _worker_record(row):\n    return _extract_record(row,*_WORKER_CONTEXT)\n\n\ndef prepare_dataset(source_root, output_root, model_path=None, processor=None, limit=None, workers=0):\n    validate_workers(workers)\n    progress(stage="preprocessing", status="running", completed=0, total=None, model=None)\n    source_root, output_root = Path(source_root).resolve(), Path(output_root).resolve()\n    if source_root == output_root or output_root.is_relative_to(source_root):\n        raise ValueError(\'Derived data must be outside source dataset\')\n    manifest = source_root/\'metadata/manifest.csv\'\n    frame = pd.read_csv(manifest,float_precision=\'round_trip\')\n    if not frame.sample_id.is_unique:\n        raise ValueError(\'Duplicate source sample IDs\')\n    if limit is not None:\n        if limit < 1: raise ValueError(\'limit must be positive\')\n        frame = frame.groupby([\'split\',\'class_id\'],sort=True).head(limit).copy()\n    owned = processor is None\n    config = processor.config if processor else PreprocessingConfig(sha256(model_path))\n    identity = {\'source_manifest_sha256\':sha256(manifest),\n                \'preprocessing_fingerprint\':config.fingerprint,\'limit_per_class_split\':limit}\n    if output_root.exists():\n        existing, report = load_dataset(output_root)\n        if report[\'identity\'] != identity:\n            raise ValueError(\'Output already contains a different dataset\')\n        progress(stage=\'preprocessing\', status=\'complete\', completed=len(frame), total=len(frame), reused=True)\n        return existing, report\n    stage = output_root.with_name(output_root.name+\'.staging\')\n    stage.mkdir(parents=True,exist_ok=True)\n    identity_path = stage/\'identity.json\'\n    if identity_path.exists() and json.loads(identity_path.read_text()) != identity:\n        raise ValueError(\'Staging identity differs; choose another output root\')\n    write_json(identity_path,identity)\n    journal = stage/\'progress.jsonl\'\n    completed = {}\n    if journal.exists():\n        lines=journal.read_bytes().splitlines(keepends=True)\n        valid_bytes=0\n        for index,line in enumerate(lines):\n            try:item=json.loads(line)\n            except (json.JSONDecodeError,UnicodeDecodeError):\n                if index!=len(lines)-1:raise ValueError(\'Corrupt preprocessing journal before last record\')\n                # A crash can interrupt the final append. Retry only that record.\n                with journal.open(\'r+b\') as stream:stream.truncate(valid_bytes)\n                break\n            completed[item[\'sample_id\']] = item\n            valid_bytes+=len(line)\n        if completed and journal.read_bytes()[-1:]!=b\'\\n\':\n            with journal.open(\'ab\') as stream:stream.write(b\'\\n\')\n    if workers < 0:raise ValueError(\'workers must be nonnegative\')\n    pool=None\n    if workers>1:\n        from concurrent.futures import ProcessPoolExecutor\n        from multiprocessing import get_context\n        pool=ProcessPoolExecutor(max_workers=workers,mp_context=get_context(\'fork\'),\n             initializer=_initialize_worker,initargs=(config,model_path,source_root,stage,processor))\n    elif processor is None:\n        processor = HandPreprocessor(config,model_path,mode=\'IMAGE\')\n    pending=[r for r in frame.to_dict(\'records\') if r[\'sample_id\'] not in completed]\n    initial=len(completed)\n    progress(stage="preprocessing", completed=initial, total=len(frame), resumed=bool(initial))\n    try:\n        items=(pool.map(_worker_record,pending,chunksize=8) if pool else\n               (_extract_record(row,source_root,stage,processor) for row in pending))\n        with journal.open(\'a\') as log:\n            for index,item in enumerate(items,initial+1):\n                log.write(json.dumps(item,allow_nan=False)+\'\\n\');log.flush()\n                completed[item[\'sample_id\']] = item\n                if index == initial+1 or index % 100 == 0 or index == len(frame):\n                    progress(stage=\'preprocessing\', completed=index, total=len(frame))\n    finally:\n        if pool is not None:pool.shutdown(wait=True,cancel_futures=True)\n        elif owned:processor.close()\n    records = [completed[s] for s in frame.sample_id]\n    # Cropping may make previously different files identical. Reject all copies\n    # rather than let duplicate crops leak across splits or retain label conflicts.\n    hashes = {}\n    for record in records:\n        if record[\'status\']==\'ok\': hashes.setdefault(record[\'image_sha256\'],[]).append(record)\n    for group in hashes.values():\n        if len(group)>1:\n            for record in group:\n                record.update(status=\'invalid\',reason=\'duplicate_derived_image\')\n                (stage/record[\'image_path\']).unlink(missing_ok=True)\n    accepted = pd.DataFrame([r for r in records if r[\'status\']==\'ok\']).drop(columns=[\'status\',\'reason\'],errors=\'ignore\')\n    rejected = pd.DataFrame([r for r in records if r[\'status\']!=\'ok\'],\n                            columns=[\'sample_id\',\'split\',\'class_id\',\'gesture\',\'status\',\'reason\'])\n    if accepted.empty:\n        raise ValueError(\'No accepted samples; inspect staging progress.jsonl\')\n    metadata = stage/\'metadata\';metadata.mkdir(exist_ok=True)\n    accepted.sort_values(\'sample_id\').to_csv(metadata/\'manifest.csv\',index=False)\n    rejected.to_csv(metadata/\'rejections.csv\',index=False)\n    report = {\'format_version\':2,\'identity\':identity,\'preprocessing\':asdict(config),\n              \'preprocessing_fingerprint\':config.fingerprint,\'class_mapping\':MAPPING,\n              \'smoke_test\':limit is not None,\n              \'counts\':{\'source\':len(frame),\'accepted\':len(accepted),\'rejected\':len(rejected)},\n              \'source_split_counts\':frame.split.value_counts().to_dict(),\n              \'split_counts\':accepted.split.value_counts().to_dict(),\n              \'class_split_counts\':accepted.groupby([\'split\',\'gesture\']).size().unstack(fill_value=0).to_dict(),\n              \'rejection_reasons\':rejected.reason.value_counts().to_dict(),\n              \'artifact_sha256\':{p.name:sha256(p) for p in [metadata/\'manifest.csv\',metadata/\'rejections.csv\']},\n              \'limits\':[\'source random split retained; not subject-wise\',\n                        \'source images were previously cropped; webcam domain differs\']}\n    write_json(metadata/\'preparation_report.json\',report)\n    # Validate complete staged artifact before publishing it.\n    load_dataset(stage)\n    output_root.parent.mkdir(parents=True,exist_ok=True)\n    os.rename(stage,output_root)\n    return load_dataset(output_root)\n\n\ndef load_dataset(root, verify_images=True):\n    root = Path(root)\n    metadata = root/\'metadata\'\n    report = json.loads((metadata/\'preparation_report.json\').read_text())\n    for name,digest in report[\'artifact_sha256\'].items():\n        if sha256(metadata/name) != digest:\n            raise ValueError(f\'Metadata hash mismatch: {name}\')\n    config = PreprocessingConfig(**report[\'preprocessing\'])\n    config.assert_matches(report[\'preprocessing_fingerprint\'])\n    frame = pd.read_csv(metadata/\'manifest.csv\',float_precision=\'round_trip\')\n    required = [\'sample_id\',\'split\',\'class_id\',\'gesture\',\'image_path\',\'image_sha256\',*LM_COLUMNS]\n    if set(required)-set(frame) or frame[required].isna().any().any():\n        raise ValueError(\'Incomplete manifest\')\n    if not np.isfinite(frame[LM_COLUMNS].to_numpy()).all():\n        raise ValueError(\'Nonfinite features\')\n    for col in [\'sample_id\',\'image_path\',\'image_sha256\']:\n        if not frame[col].is_unique: raise ValueError(f\'Duplicate {col}\')\n    if (frame.split.value_counts().to_dict() != report[\'split_counts\']\n            or set(frame.split) != set(SPLITS) or report[\'class_mapping\'] != MAPPING):\n        raise ValueError(\'Split counts or mapping mismatch\')\n    for split in SPLITS:\n        if set(frame.loc[frame.split.eq(split),\'class_id\']) != set(range(6)):\n            raise ValueError(f\'Split {split} is missing classes\')\n    for c,g in enumerate(GESTURES):\n        if not frame.loc[frame.class_id.eq(c),\'gesture\'].eq(g).all():\n            raise ValueError(\'Invalid class mapping\')\n    if not set(frame.class_id).issubset(range(6)):\n        raise ValueError(\'Invalid class ID\')\n    if verify_images:\n        for row in frame.itertuples():\n            if sha256(safe_path(root,row.image_path)) != row.image_sha256:\n                raise ValueError(f\'Image hash mismatch: {row.image_path}\')\n    return frame.sort_values(\'sample_id\').reset_index(drop=True), report\n'
exec(compile(_source, 'handgesture/data.py', 'exec'), _module.__dict__)

In [ ]:
from handgesture.data import prepare_dataset
from handgesture.runtime import stage
DATA_ROOT = Path(DATA_ROOT) if DATA_ROOT else find_dataset(False)
DETECTOR_PATH = Path(DETECTOR_PATH) if DETECTOR_PATH else find_asset('hand_landmarker.task')
with stage('preprocessing'):
    frame, report = prepare_dataset(DATA_ROOT, OUTPUT_ROOT/'data_v2', DETECTOR_PATH,
                                     limit=8 if SMOKE_TEST else None, workers=NUM_WORKERS)
print(report['counts'], report['split_counts'])
